# A3.1 · Sandboxing is the perimeter

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

---

**Risk.** Tier chosen by convenience, not by action class.

**Control.** Process → container → microVM → ephemeral workstation → air-gapped runner, chosen deliberately.

**This lab.** Measure escape surface across three sandbox tiers.

| | |
|---|---|
| Open-source tooling | gVisor, Firecracker, Docker |
| Open-weight models | Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A3.1"))

For an agent, the sandbox is the perimeter — because intent is not a control you own, and the prompt can be fully compromised while every containment property still holds.

In [ ]:
from cybercommons import sandbox

box = sandbox.default_sandbox()
attempts = [
    ("read_file",  "/work/src/app.py"),
    ("read_file",  "/work/../../root/.ssh/id_rsa"),
    ("read_file",  "/work/.env"),
    ("http_get",   "https://api.github.com/repos/x/y"),
    ("http_get",   "http://169.254.169.254/latest/meta-data/iam/"),
    ("http_get",   "https://exfil.example.com/collect"),
    ("run_shell",  ""),
    ("delete_repo", ""),
]
for tool, target in attempts:
    print(box.call(tool, target))
print("\n", box.summary())

Assume the prompt is entirely under attacker control. Every one of these was *requested*; six were refused anyway. That is what it means for containment to be the perimeter.

### Expect

Two calls succeed (the workspace read and the allowlisted host). Six are denied, and `summary()` lists the distinct reasons.

### Your turn

Which single denial in that list would you most regret losing? Now check whether your production agent actually has it.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A3.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*